In [26]:
import requests
import pandas as pd
import re
import math

# API키 입력!
NAVER_CLIENT_ID = "-"
NAVER_CLIENT_SECRET = "-"
KAKAO_API_KEY = "-"

PNU_LAT = 35.2323
PNU_LNG = 129.0847

def get_naver_food():
    url = "https://openapi.naver.com/v1/search/local.json"
    headers = {
        "X-Naver-Client-Id": NAVER_CLIENT_ID,
        "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
    }
    all_results = []
    queries = ["부산대 맛집", "부산대 한식", "부산대 일식", "부산대 중식", "부산대 양식",
               "부산대 치킨", "부산대 피자", "부산대 분식", "부산대 고기", "부산대 족발",
               "부산대 찜탕", "부산대 돈가스", "부산대 아시안", "부산대 패스트푸드"]

    seen = set()  # 중복 제거용
    for query in queries:
        params = {"query": query, "display": 5, "sort": "comment"}
        res = requests.get(url, headers=headers, params=params)
        items = res.json().get("items", [])
        for item in items:
            if item["title"] not in seen:
                seen.add(item["title"])
                all_results.append(item)

    return all_results


def get_naver_cafe():
    url = "https://openapi.naver.com/v1/search/local.json"
    headers = {
        "X-Naver-Client-Id": NAVER_CLIENT_ID,
        "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
    }
    all_results = []
    queries = ["부산대 카페", "부산대 커피", "부산대 디저트", "부산대 베이커리", "부산대 브런치"]

    seen = set()
    for query in queries:
        params = {"query": query, "display": 5, "sort": "comment"}
        res = requests.get(url, headers=headers, params=params)
        items = res.json().get("items", [])
        for item in items:
            if item["title"] not in seen:
                seen.add(item["title"])
                all_results.append(item)

    print("네이버 카페 총:", len(all_results), "개")
    return all_results

def get_kakao_food(lat=PNU_LAT, lng=PNU_LNG, radius=1000):
    url = "https://dapi.kakao.com/v2/local/search/category.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    all_results = []
    seen = set()  # 중복 제거용

    for page in range(1, 8):
        params = {
            "category_group_code": "FD6",
            "x": lng,
            "y": lat,
            "radius": radius,
            "sort": "distance",
            "size": 15,
            "page": page
        }
        res = requests.get(url, headers=headers, params=params)
        data = res.json()
        docs = data.get("documents", [])

        for doc in docs:
            if doc["place_name"] not in seen:
                seen.add(doc["place_name"])
                all_results.append(doc)

        if data["meta"]["is_end"]:
            break

    print("카카오 밥집 총:", len(all_results), "개")
    return all_results

def get_kakao_cafe(lat=PNU_LAT, lng=PNU_LNG, radius=1000):
    url = "https://dapi.kakao.com/v2/local/search/category.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    all_results = []
    seen = set()

    for page in range(1, 8):
        params = {
            "category_group_code": "CE7",
            "x": lng,
            "y": lat,
            "radius": radius,
            "sort": "distance",
            "size": 15,
            "page": page
        }
        res = requests.get(url, headers=headers, params=params)
        data = res.json()
        docs = data.get("documents", [])

        for doc in docs:
            if doc["place_name"] not in seen:
                seen.add(doc["place_name"])
                all_results.append(doc)

        if data["meta"]["is_end"]:
            break

    print("카카오 카페 총:", len(all_results), "개")
    return all_results


def calc_walk_time(distance_m):
    if not distance_m:
        return "-"

    real_distance = int(distance_m) * 1.3
    minutes = real_distance / 67

    if minutes < 1:
        return "1분 이내"

    return f"약 {math.ceil(minutes)}분"

naver_food = get_naver_food()
naver_cafe = get_naver_cafe()
kakao_food = get_kakao_food()
kakao_cafe = get_kakao_cafe()


네이버 카페 총: 13 개
카카오 밥집 총: 45 개
카카오 카페 총: 45 개


In [27]:
food_naver_df = pd.DataFrame([{
    "이름": re.sub('<.*?>', '', r["title"]),
    "주소": r["roadAddress"],
    "카테고리": r["category"].split(">")[1].strip() if ">" in r["category"] else r["category"],
    "거리(m)": "-",
    "도보시간": "-",
    "출처": "네이버"
} for r in naver_food if "베이커리" not in r["category"] and "카페" not in r["category"]])

exclude = ["샐러드", "간식", "베이커리", "제과", "도시락", "술집", "호프", "야식"]

food_kakao_df = pd.DataFrame([{
    "이름": r["place_name"],
    "주소": r["road_address_name"],
    "카테고리": r["category_name"].split(">")[1].strip() if ">" in r["category_name"] else r["category_name"],
    "거리(m)": r["distance"],
    "도보시간": calc_walk_time(r["distance"]),
    "출처": "카카오"
} for r in kakao_food if not any(ex in r["category_name"] for ex in exclude)])


food_df = pd.concat([food_naver_df, food_kakao_df], ignore_index=True)
food_df

import ipywidgets as widgets
from IPython.display import display, clear_output

def get_main_category(category):
    category = category.replace(" ", "")
    if "치킨" in category: return "치킨"
    elif "패스트푸드" in category or "햄버거" in category: return "패스트푸드"
    elif "돈가스" in category or "회" in category or "초밥" in category or "일식" in category: return "돈까스·회"
    elif "베트남" in category or "아시아" in category or "인도" in category or "태국" in category: return "아시안"
    elif "족발" in category or "보쌈" in category: return "족발·보쌈"
    elif "피자" in category: return "피자"
    elif "찜" in category or "탕" in category or "순대" in category or "국밥" in category: return "찜·탕"
    elif "중식" in category or "중국" in category: return "중식"
    elif "분식" in category: return "분식"
    elif "한식" in category: return "한식"
    elif "고기" in category or "구이" in category or "삼겹" in category: return "고기"
    elif "양식" in category or "스테이크" in category or "파스타" in category: return "양식"
    else: return "기타"


food_df["대분류"] = food_df["카테고리"].apply(get_main_category)

sort_dropdown1 = widgets.Dropdown(
    options=["거리순", "카테고리별"],
    description="밥집 정렬:",
    value="거리순"
)
output1 = widgets.Output()

def update_food(change):
    with output1:
        clear_output(wait=True)
        if sort_dropdown1.value == "거리순":
            result = food_df.copy()
            result["거리정렬"] = result["거리(m)"].apply(lambda x: int(x) if x != "-" else 9999)
            display(result.sort_values("거리정렬").drop(columns=["거리정렬"]).reset_index(drop=True))
        else:
            display(food_df.sort_values("대분류").reset_index(drop=True))

sort_dropdown1.observe(update_food, names="value")
display(sort_dropdown1, output1)
update_food(None)


Dropdown(description='밥집 정렬:', options=('거리순', '카테고리별'), value='거리순')

Output()

In [34]:
exclude_cafe_kakao = ["키즈", "보드", "스파게티", "파스타", "여가", "실내놀이", "방탈출", "노래", "당구", "스크린"]
exclude_cafe_naver = ["키즈", "보드", "스파게티", "파스타", "여가", "실내놀이"]

cafe_naver_df = pd.DataFrame([{
    "이름": re.sub('<.*?>', '', r["title"]),
    "주소": r["roadAddress"],
    "카테고리": r["category"].split(">")[1].strip() if ">" in r["category"] else r["category"],
    "거리(m)": "-",
    "도보시간": "-",
    "출처": "네이버"
} for r in naver_cafe if not any(ex in r["category"] for ex in exclude_cafe_naver)])

cafe_kakao_df = pd.DataFrame([{
    "이름": r["place_name"],
    "주소": r["road_address_name"],
    "카테고리": r["category_name"].split(">")[1].strip() if ">" in r["category_name"] else r["category_name"],
    "거리(m)": r["distance"],
    "도보시간": calc_walk_time(r["distance"]),
    "출처": "카카오"
} for r in kakao_cafe if not any(ex in r["category_name"] for ex in exclude_cafe_kakao)])

cafe_df = pd.concat([cafe_naver_df, cafe_kakao_df], ignore_index=True)

cafe_df["대분류"] = cafe_df["카테고리"].apply(get_main_category)

sort_dropdown2 = widgets.Dropdown(
    options=["거리순", "카테고리별"],
    description="카페 정렬:",
    value="거리순"
)
output2 = widgets.Output()

def update_cafe(change):
    with output2:
        clear_output(wait=True)
        if sort_dropdown2.value == "거리순":
            result = cafe_df.copy()
            result["거리정렬"] = result["거리(m)"].apply(lambda x: int(x) if x != "-" else 9999)
            display(result.sort_values("거리정렬").drop(columns=["거리정렬"]).reset_index(drop=True))
        else:
            display(cafe_df.sort_values("대분류").reset_index(drop=True))

sort_dropdown2.observe(update_cafe, names="value")
display(sort_dropdown2, output2)
update_cafe(None)



Dropdown(description='카페 정렬:', options=('거리순', '카테고리별'), value='거리순')

Output()

In [ ]:
from google.colab import files

# 밥집이랑 카페 각각 다른 시트로 저장
with pd.ExcelWriter("부산대_식당목록.xlsx") as writer:
    food_df.to_excel(writer, sheet_name="밥집", index=False)
    cafe_df.to_excel(writer, sheet_name="카페", index=False)

files.download("부산대_식당목록.xlsx")
print("저장 완료!")